# Predicting F1 Pit Stops — Optimal Pipeline (v5)
**Playground Series S6E5** | Target: PitNextLap | Metric: ROC-AUC

**v4 → v5** (break the 0.95 barrier):
1. **sklearn MLPClassifier** as 4th model — architectural diversity from GBDTs.
2. **Per-year AUC + calibration** — catches the 2023 regime anomaly.
3. **Logistic stacker** with coefficient inspection — auto-drops negative-coef models.
4. **Isotonic calibration** of final blend — preserves AUC, recovers probability scale.
5. OOF arrays cached to oof/ — re-blending is instant.

In [ ]:
import os, gc, warnings, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import seaborn as sns
from scipy.stats import rankdata
from scipy.special import expit, logit
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import calibration_curve
import lightgbm as lgb
from catboost import CatBoostClassifier
import xgboost as xgb

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid", context="notebook")
SEED, N_FOLDS = 42, 5
DATA = Path("data")
OOF  = Path("oof"); OOF.mkdir(exist_ok=True)

## 1. Load

In [ ]:
train = pd.read_csv(DATA / 'train.csv')
test  = pd.read_csv(DATA / 'test.csv')
train.rename(columns={'LapTime (s)': 'LapTime'}, inplace=True)
test.rename(columns={'LapTime (s)': 'LapTime'}, inplace=True)
TARGET, ID = 'PitNextLap', 'id'
print(f'train: {train.shape}  test: {test.shape}  pos rate: {train[TARGET].mean():.4f}')

## 2. EDA

In [ ]:
# 2a. Target rate by Compound
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
by_comp = train.groupby('Compound')[TARGET].agg(['mean','count']).sort_values('count', ascending=False)
by_comp['mean'].plot.bar(ax=axes[0], color='steelblue', edgecolor='k')
axes[0].set_title('Pit-next-lap rate by Compound'); axes[0].set_ylabel('P(pit)'); axes[0].set_xlabel('')
by_comp['count'].plot.bar(ax=axes[1], color='gray', edgecolor='k')
axes[1].set_title('Sample count by Compound'); axes[1].set_ylabel('rows'); axes[1].set_xlabel('')
plt.tight_layout(); plt.show()
by_comp

In [ ]:
# 2b. Pit-rate per year — check for 2023 regime anomaly (Ivan's lesson)
yr_stats = (train.groupby('Year')
            .agg(rows=('id','count'), pit_rate=(TARGET,'mean'))
            .assign(pit_rate_pct=lambda d: d['pit_rate'].mul(100).round(2)))
print(yr_stats)
fig, ax = plt.subplots(figsize=(7, 3.5))
yr_stats['pit_rate'].plot.bar(ax=ax, color='steelblue', edgecolor='k')
for i, v in enumerate(yr_stats['pit_rate']):
    ax.text(i, v+0.005, f'{v:.4f}', ha='center', fontsize=9)
ax.set_title('Pit-next-lap rate by Year (uneven → year acts as a regime feature)')
ax.set_ylabel('P(pit)'); plt.tight_layout(); plt.show()

In [ ]:
# 2c. Pit hazard curves per compound
fig, ax = plt.subplots(figsize=(10, 4.5))
for comp in ['SOFT','MEDIUM','HARD','INTERMEDIATE','WET']:
    s = train[train['Compound']==comp]
    if len(s) < 100: continue
    bins = np.arange(0, s['TyreLife'].max()+2, 2)
    s = s.assign(tl_bin=pd.cut(s['TyreLife'], bins))
    rate = s.groupby('tl_bin')[TARGET].mean()
    centers = [b.mid for b in rate.index]
    ax.plot(centers, rate.values, marker='o', label=comp, lw=2)
ax.set_xlabel('TyreLife (laps)'); ax.set_ylabel('P(pit next lap)')
ax.set_title('Pit-hazard curves'); ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# 2d. Compound × TyreLife heatmap
h = train.copy(); h['tl_bin'] = pd.cut(h['TyreLife'], bins=np.arange(0, 50, 3))
pivot = h.groupby(['Compound','tl_bin'])[TARGET].mean().unstack()
fig, ax = plt.subplots(figsize=(13, 3.5))
sns.heatmap(pivot, cmap='RdYlBu_r', cbar_kws={'label':'P(pit)'}, ax=ax)
ax.set_title('P(pit next lap) — Compound × TyreLife'); ax.set_xlabel('TyreLife bin'); ax.set_ylabel('')
plt.tight_layout(); plt.show()

In [ ]:
# 2e. Race-track avatars
def make_track_xy(seed, n=400):
    rng = np.random.default_rng(seed)
    theta = np.linspace(0, 2*np.pi, n, endpoint=False)
    r = np.ones_like(theta)
    for k in range(2, 8):
        amp = rng.uniform(-0.18, 0.18) / (k * 0.6)
        phase = rng.uniform(0, 2*np.pi)
        r = r + amp * np.cos(k*theta + phase)
    return r*np.cos(theta), r*np.sin(theta), theta
N_RP_BINS = 60
tmp = train.copy()
tmp['rp_bin'] = (tmp['RaceProgress'].clip(0, 1-1e-6) * N_RP_BINS).astype(int)
race_rate = tmp.groupby(['Race','rp_bin'])[TARGET].mean().unstack(fill_value=np.nan)
races = (race_rate.max(axis=1) - race_rate.min(axis=1)).sort_values(ascending=False).head(9).index.tolist()
fig, axes = plt.subplots(3, 3, figsize=(13, 13))
vmin, vmax = 0.0, float(np.nanpercentile(race_rate.values, 99))
for ax, race in zip(axes.flat, races):
    x, y_, theta = make_track_xy(seed=abs(hash(race)) % (2**32))
    bin_idx = ((theta / (2*np.pi)) * N_RP_BINS).astype(int)
    rates = race_rate.loc[race].reindex(range(N_RP_BINS)).values
    rate_per_pt = rates[bin_idx]
    pts = np.array([x, y_]).T.reshape(-1, 1, 2)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
    lc = LineCollection(segs, cmap='RdYlBu_r', linewidth=10, norm=plt.Normalize(vmin, vmax))
    lc.set_array(rate_per_pt[:-1]); ax.add_collection(lc)
    ax.plot(x[0], y_[0], 'ks', markersize=7); ax.text(x[0]+0.05, y_[0]+0.05, 'S/F', fontsize=8)
    ax.set_xlim(-1.35, 1.35); ax.set_ylim(-1.35, 1.35); ax.set_aspect('equal'); ax.axis('off')
    n_rows = int((train['Race']==race).sum())
    pit_rate = train.loc[train['Race']==race, TARGET].mean()
    ax.set_title(f'{race}\nrows={n_rows:,}  pit rate={pit_rate:.3f}', fontsize=10)
fig.suptitle('Pit risk painted onto race-progress track avatars', fontsize=14, y=1.0)
fig.colorbar(lc, ax=axes, shrink=0.5, label='P(pit next lap)'); plt.show()

In [ ]:
# 2f. Stint length distribution per compound
stint_lens = train.groupby(['Race','Year','Driver','Stint','Compound']).size().reset_index(name='laps')
fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(data=stint_lens, x='Compound', y='laps',
            order=['SOFT','MEDIUM','HARD','INTERMEDIATE','WET'], ax=ax)
ax.set_title('Stint length by compound'); ax.set_ylabel('laps in stint')
plt.tight_layout(); plt.show()

## 3. Feature Engineering

In [ ]:
train['is_test'] = 0;  test['is_test'] = 1;  test[TARGET] = np.nan
full = pd.concat([train, test], ignore_index=True)
full = full.sort_values(['Race','Year','Driver','LapNumber']).reset_index(drop=True)
G    = ['Race','Year','Driver']
GS   = ['Race','Year','Driver','Stint']
RACE = ['Race','Year']

g = full.groupby(G, sort=False)
for col in ['TyreLife','LapTime','Position','Stint','Compound',
            'PitStop','Cumulative_Degradation','LapTime_Delta',
            'Position_Change','LapNumber']:
    full[f'{col}_prev'] = g[col].shift(1)
    full[f'{col}_next'] = g[col].shift(-1)
for col in ['TyreLife','LapTime','Stint','Compound','PitStop','LapNumber']:
    full[f'{col}_prev2'] = g[col].shift(2)
    full[f'{col}_next2'] = g[col].shift(-2)

full['lap_gap_prev']  = full['LapNumber'] - full['LapNumber_prev']
full['lap_gap_next']  = full['LapNumber_next']  - full['LapNumber']
full['lap_gap_next2'] = full['LapNumber_next2'] - full['LapNumber']
full['lap_gap_prev2'] = full['LapNumber'] - full['LapNumber_prev2']

full['tyre_life_delta_next']  = full['TyreLife_next']  - full['TyreLife']
full['tyre_life_delta_next2'] = full['TyreLife_next2'] - full['TyreLife']
full['tyre_life_delta_prev']  = full['TyreLife']       - full['TyreLife_prev']
full['stint_change_next']     = (full['Stint_next']  > full['Stint']).astype('float')
full['stint_change_next2']    = (full['Stint_next2'] > full['Stint']).astype('float')
full['stint_change_prev']     = (full['Stint'] > full['Stint_prev']).astype('float')
full['compound_change_next']  = (full['Compound_next']  != full['Compound']).astype('float')
full['compound_change_next2'] = (full['Compound_next2'] != full['Compound']).astype('float')
full['just_pitted']           = (full['stint_change_prev'] == 1).astype('float')

full['oracle_pit_next'] = np.where(full['lap_gap_next'] == 1, full['PitStop_next'], np.nan)
full['oracle_gap2_pit'] = np.where(
    (full['lap_gap_next'] == 2) & (full['Stint_next'] > full['Stint']),
    1.0, np.nan)

full['laptime_diff_prev']  = full['LapTime'] - full['LapTime_prev']
full['laptime_diff_next']  = full['LapTime_next'] - full['LapTime']
full['laptime_diff_prev2'] = full['LapTime'] - full['LapTime_prev2']
full['pos_diff_prev']      = full['Position'] - full['Position_prev']
full['pos_diff_next']      = full['Position_next'] - full['Position']
full['laptime_roll3_mean'] = g['LapTime'].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
full['laptime_vs_recent']  = full['LapTime'] - full['laptime_roll3_mean']

gs = full.groupby(GS, sort=False)
full['stint_lap_min']      = gs['LapNumber'].transform('min')
full['stint_lap_max']      = gs['LapNumber'].transform('max')
full['stint_lap_count']    = gs['LapNumber'].transform('count')
full['stint_tyre_max']     = gs['TyreLife'].transform('max')
full['stint_progress']     = (full['LapNumber'] - full['stint_lap_min']) / (full['stint_lap_max'] - full['stint_lap_min'] + 1)
full['laps_left_in_stint'] = full['stint_lap_max'] - full['LapNumber']
full['tyre_left_in_stint'] = full['stint_tyre_max'] - full['TyreLife']
full['stint_laptime_mean'] = gs['LapTime'].transform('mean')
full['laptime_vs_stint']   = full['LapTime'] - full['stint_laptime_mean']
full['stint_deg_max']      = gs['Cumulative_Degradation'].transform('max')
full['deg_pct_of_max']     = full['Cumulative_Degradation'] / (full['stint_deg_max'] + 1e-6)
full['is_last_obs_of_stint'] = (full['LapNumber'] == full['stint_lap_max']).astype('float')

tr_mask = full['is_test'] == 0
cm = full[tr_mask].groupby('Compound')['TyreLife'].quantile(0.95).rename('compound_typical_max')
full = full.merge(cm, on='Compound', how='left')
full['tyre_life_normalised'] = full['TyreLife'] / (full['compound_typical_max'] + 1e-6)
full['tyre_life_over_max']   = (full['TyreLife'] > full['compound_typical_max']).astype('float')
csl = full[tr_mask].groupby('Compound')['stint_lap_count'].mean().rename('compound_avg_stint_len')
full = full.merge(csl, on='Compound', how='left')
full['stint_len_vs_compound_avg'] = full['stint_lap_count'] - full['compound_avg_stint_len']

rl = full.groupby(['Race','Year','LapNumber'])
full['n_drivers_this_lap']     = rl['Driver'].transform('count')
full['mean_position_this_lap'] = rl['Position'].transform('mean')
full['mean_laptime_this_lap']  = rl['LapTime'].transform('mean')
full['laptime_vs_field']       = full['LapTime'] - full['mean_laptime_this_lap']
full['field_pitrate_this_lap'] = rl['PitStop'].transform('mean')
rlc = full.groupby(['Race','Year','LapNumber','Compound'])
full['n_same_compound_this_lap'] = rlc['Driver'].transform('count')
full['frac_same_compound']       = full['n_same_compound_this_lap'] / full['n_drivers_this_lap']
ru = full.groupby(['Race','Year','LapNumber'])['PitStop'].sum().reset_index().rename(columns={'PitStop':'race_pits_this_lap'})
ru = ru.sort_values(['Race','Year','LapNumber'])
ru['race_pits_next_lap'] = ru.groupby(['Race','Year'])['race_pits_this_lap'].shift(-1)
full = full.merge(ru[['Race','Year','LapNumber','race_pits_next_lap']], on=['Race','Year','LapNumber'], how='left')

full['stints_done_so_far'] = full['Stint'] - 1
full['race_total_laps']    = full.groupby(RACE)['LapNumber'].transform('max')
full['laps_left_in_race']  = full['race_total_laps'] - full['LapNumber']
full['frac_laps_left']     = full['laps_left_in_race'] / (full['race_total_laps'] + 1e-6)
full['driver_pits_so_far'] = g['PitStop'].cumsum() - full['PitStop']

CAT_COLS = ['Driver','Compound','Race','Compound_prev','Compound_next','Compound_prev2','Compound_next2']
for c in CAT_COLS:
    full[c] = full[c].astype('category')

train_fe = full[full['is_test']==0].copy()
test_fe  = full[full['is_test']==1].copy()
print(f'train_fe: {train_fe.shape}  test_fe: {test_fe.shape}')

## 4. OOF target encoding

In [ ]:
TE_GROUPS = [('Driver',), ('Race',), ('Compound','Stint'),
             ('Driver','Compound'), ('Race','Compound'), ('Driver','Race')]
ALPHA = 20.0
y = train_fe[TARGET].astype(int).values
global_mean = y.mean()
folds = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(train_fe, y))

def te_encode(cols, train_df, test_df, y, folds, alpha=ALPHA):
    cols = list(cols)
    key = 'te_' + '_x_'.join(cols)
    oof = np.full(len(train_df), global_mean, dtype=np.float32)
    for tr_idx, va_idx in folds:
        sub = train_df.iloc[tr_idx][cols].copy()
        sub['_y'] = y[tr_idx]
        agg = sub.groupby(cols, observed=True)['_y'].agg(['sum','count'])
        agg['te'] = (agg['sum'] + alpha * global_mean) / (agg['count'] + alpha)
        merged = (train_df.iloc[va_idx][cols]
                  .merge(agg.reset_index()[cols+['te']], on=cols, how='left'))
        oof[va_idx] = merged['te'].fillna(global_mean).astype(np.float32).values
    sub = train_df[cols].copy(); sub['_y'] = y
    agg = sub.groupby(cols, observed=True)['_y'].agg(['sum','count'])
    agg['te'] = (agg['sum'] + alpha * global_mean) / (agg['count'] + alpha)
    merged = test_df[cols].merge(agg.reset_index()[cols+['te']], on=cols, how='left')
    return key, oof, merged['te'].fillna(global_mean).astype(np.float32).values

for grp in TE_GROUPS:
    key, oof_te, pred_te = te_encode(grp, train_fe, test_fe, y, folds)
    train_fe[key] = oof_te; test_fe[key] = pred_te
    print(f'  {key}: mean={oof_te.mean():.4f}  std={oof_te.std():.4f}')

drop_cols = [ID, TARGET, 'is_test']
feature_cols = [c for c in train_fe.columns if c not in drop_cols]
X    = train_fe[feature_cols]
X_te = test_fe[feature_cols]
print(f'features: {len(feature_cols)}  train: {X.shape}  test: {X_te.shape}')

## 5. Model training
Each model auto-caches OOF + test predictions to `oof/`. Re-runs load from cache instantly.

In [ ]:
def cached(name, train_fn):
    path = OOF / f'{name}.npz'
    if path.exists():
        d = np.load(path)
        oof, pred = d['oof'], d['pred']
        print(f'{name} OOF (cached): {roc_auc_score(y, oof):.5f}')
        return oof, pred
    oof, pred = train_fn()
    np.savez(path, oof=oof, pred=pred)
    return oof, pred

In [ ]:
# 5a. LightGBM
def train_lgb():
    params = dict(objective='binary', metric='auc',
                  learning_rate=0.03, num_leaves=511, min_child_samples=30,
                  feature_fraction=0.85, bagging_fraction=0.85, bagging_freq=1,
                  lambda_l2=2.0, verbose=-1, seed=SEED, n_jobs=-1)
    oof, pred = np.zeros(len(X)), np.zeros(len(X_te))
    for f, (tr, va) in enumerate(folds):
        t0 = time.time()
        m = lgb.train(params,
                      lgb.Dataset(X.iloc[tr], y[tr], categorical_feature=CAT_COLS),
                      6000,
                      valid_sets=[lgb.Dataset(X.iloc[va], y[va], categorical_feature=CAT_COLS)],
                      callbacks=[lgb.early_stopping(150), lgb.log_evaluation(0)])
        oof[va] = m.predict(X.iloc[va], num_iteration=m.best_iteration)
        pred   += m.predict(X_te, num_iteration=m.best_iteration) / N_FOLDS
        print(f'  fold {f+1}: AUC={roc_auc_score(y[va], oof[va]):.5f}  {time.time()-t0:.1f}s')
    return oof, pred
oof_lgb, pred_lgb = cached('lgb', train_lgb)
auc_lgb = roc_auc_score(y, oof_lgb)

In [ ]:
# 5b. Bagged CatBoost (3 configs)
CB_CONFIGS = [
    dict(seed=42,   depth=8, learning_rate=0.05, l2_leaf_reg=3.0, bagging_temperature=0.8, random_strength=1.0),
    dict(seed=7,    depth=7, learning_rate=0.06, l2_leaf_reg=2.0, bagging_temperature=1.0, random_strength=1.5),
    dict(seed=2023, depth=9, learning_rate=0.04, l2_leaf_reg=4.0, bagging_temperature=0.6, random_strength=0.8),
]
X_cb, X_te_cb = X.copy(), X_te.copy()
for c in CAT_COLS:
    X_cb[c]    = X_cb[c].astype(str).fillna('NA')
    X_te_cb[c] = X_te_cb[c].astype(str).fillna('NA')

def train_cb_single(cfg, ci):
    oof, pred = np.zeros(len(X)), np.zeros(len(X_te))
    for f, (tr, va) in enumerate(folds):
        t0 = time.time()
        m = CatBoostClassifier(iterations=4000,
            **{k:v for k,v in cfg.items() if k!='seed'},
            random_seed=cfg['seed'], eval_metric='AUC',
            cat_features=CAT_COLS, early_stopping_rounds=120,
            verbose=0, task_type='CPU')
        m.fit(X_cb.iloc[tr], y[tr], eval_set=(X_cb.iloc[va], y[va]), use_best_model=True)
        oof[va] = m.predict_proba(X_cb.iloc[va])[:,1]
        pred   += m.predict_proba(X_te_cb)[:,1] / N_FOLDS
        print(f'    fold {f+1}: AUC={roc_auc_score(y[va], oof[va]):.5f}  {time.time()-t0:.1f}s')
    return oof, pred

oof_cb_all, pred_cb_all = [], []
for ci, cfg in enumerate(CB_CONFIGS):
    print(f'\nCB[{ci+1}/{len(CB_CONFIGS)}] cfg={cfg}')
    o, p = cached(f'cb_{ci}', lambda cfg=cfg, ci=ci: train_cb_single(cfg, ci))
    oof_cb_all.append(o); pred_cb_all.append(p)

def to_rank(x, eps=1e-6):
    r = rankdata(x) / len(x); return np.clip(r, eps, 1-eps)
def logit_rank(preds, weights):
    z = sum(w * logit(to_rank(p)) for w, p in zip(weights, preds))
    return expit(z / sum(weights))

auc_cb_each = [roc_auc_score(y, o) for o in oof_cb_all]
w_cb = np.array(auc_cb_each) ** 2
oof_cb  = logit_rank(oof_cb_all, w_cb)
pred_cb = logit_rank(pred_cb_all, w_cb)
auc_cb  = roc_auc_score(y, oof_cb)
print(f'\nCB individual OOF: {[f"{a:.5f}" for a in auc_cb_each]}')
print(f'CB BAGGED OOF AUC: {auc_cb:.5f}')

In [ ]:
# 5c. XGBoost (diversity check — Ivan's finding: usually too correlated with LGB)
X_xgb, X_te_xgb = X.copy(), X_te.copy()
for c in CAT_COLS:
    X_xgb[c] = X_xgb[c].astype('category')
    X_te_xgb[c] = X_te_xgb[c].astype('category')

def train_xgb():
    params = dict(objective='binary:logistic', eval_metric='auc',
                  tree_method='hist', enable_categorical=True,
                  learning_rate=0.05, max_depth=8,
                  min_child_weight=20, subsample=0.85, colsample_bytree=0.85,
                  reg_lambda=2.0, seed=SEED, n_jobs=-1, verbosity=0)
    oof, pred = np.zeros(len(X)), np.zeros(len(X_te))
    dte = xgb.DMatrix(X_te_xgb, enable_categorical=True)
    for f, (tr, va) in enumerate(folds):
        t0 = time.time()
        m = xgb.train(params,
                      xgb.DMatrix(X_xgb.iloc[tr], label=y[tr], enable_categorical=True),
                      num_boost_round=4000,
                      evals=[(xgb.DMatrix(X_xgb.iloc[va], label=y[va], enable_categorical=True),'va')],
                      early_stopping_rounds=120, verbose_eval=0)
        dva = xgb.DMatrix(X_xgb.iloc[va], enable_categorical=True)
        oof[va] = m.predict(dva, iteration_range=(0, m.best_iteration+1))
        pred   += m.predict(dte, iteration_range=(0, m.best_iteration+1)) / N_FOLDS
        print(f'  fold {f+1}: AUC={roc_auc_score(y[va], oof[va]):.5f}  {time.time()-t0:.1f}s')
    return oof, pred

oof_xgb, pred_xgb = cached('xgb', train_xgb)
auc_xgb = roc_auc_score(y, oof_xgb)

## 6. Neural Net (sklearn MLP)
Feed-forward NN as architectural diversity from GBDTs. We use sklearn  (battle-tested, stable on CPU) since RealMLP segfaults on our 100+ engineered features. Standardized inputs, label-encoded categoricals.

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPClassifier

# Prep: label-encode cats, scale numerics, hard-clip outliers
X_rm    = X.copy()
X_te_rm = X_te.copy()
for c in CAT_COLS:
    le = LabelEncoder()
    combined = pd.concat([X_rm[c].astype(str), X_te_rm[c].astype(str)])
    le.fit(combined)
    X_rm[c]    = le.transform(X_rm[c].astype(str)).astype(np.float32)
    X_te_rm[c] = le.transform(X_te_rm[c].astype(str)).astype(np.float32)

num_cols = [c for c in feature_cols if c not in CAT_COLS]
for c in num_cols:
    if X_rm[c].isnull().any() or X_te_rm[c].isnull().any():
        med = X_rm[c].median()
        X_rm[c]    = X_rm[c].fillna(med)
        X_te_rm[c] = X_te_rm[c].fillna(med)

scaler = StandardScaler()
all_cols = list(X_rm.columns)
X_rm[all_cols]    = np.clip(scaler.fit_transform(X_rm[all_cols].astype(np.float64)), -10, 10)
X_te_rm[all_cols] = np.clip(scaler.transform(X_te_rm[all_cols].astype(np.float64)), -10, 10)

def train_mlp():
    oof, pred = np.zeros(len(X)), np.zeros(len(X_te))
    for f, (tr, va) in enumerate(folds):
        t0 = time.time()
        m = MLPClassifier(
            hidden_layer_sizes=(256, 128, 64),
            activation="relu", solver="adam",
            alpha=1e-4, batch_size=2048,
            learning_rate_init=1e-3, max_iter=40,
            early_stopping=True, validation_fraction=0.1,
            n_iter_no_change=5, random_state=SEED + f, verbose=False)
        m.fit(X_rm.iloc[tr].values, y[tr])
        oof[va] = m.predict_proba(X_rm.iloc[va].values)[:, 1]
        pred   += m.predict_proba(X_te_rm.values)[:, 1] / N_FOLDS
        print(f"  fold {f+1}: AUC={roc_auc_score(y[va], oof[va]):.5f}  iter={m.n_iter_}  {time.time()-t0:.1f}s")
    return oof, pred

oof_rm, pred_rm = cached("mlp", train_mlp)
auc_rm = roc_auc_score(y, oof_rm)


## 7. Diagnostics

In [ ]:
# 7a. Inter-model OOF correlation — Ivan's diagnostic for blend value
# Pearson + Spearman; Ivan flagged ρ(LGB,XGB)=0.981 as too-correlated.
oof_df = pd.DataFrame({'lgb':oof_lgb, 'cb_bag':oof_cb, 'xgb':oof_xgb, 'realmlp':oof_rm})
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.heatmap(oof_df.corr('pearson'),  annot=True, fmt='.4f', cmap='viridis', cbar=False, ax=axes[0])
axes[0].set_title('Pearson')
sns.heatmap(oof_df.corr('spearman'), annot=True, fmt='.4f', cmap='viridis', cbar=False, ax=axes[1])
axes[1].set_title('Spearman')
fig.suptitle('OOF prediction correlation across models'); plt.tight_layout(); plt.show()
print('\nIvan rule: corr > 0.98 means the second model is just noisy copy → blend hurts.')

In [ ]:
# 7b. Per-year AUC + calibration (Ivan's lesson on 2023 anomaly)
yrs = sorted(train_fe['Year'].unique())
rows = []
for yr in yrs:
    m = train_fe['Year'].values == yr
    yt = y[m]
    if yt.sum() == 0 or yt.sum() == len(yt): continue
    rows.append({
        'year': yr, 'rows': int(m.sum()), 'pit_rate': float(yt.mean()),
        'cb_auc':   roc_auc_score(yt, oof_cb[m]),
        'lgb_auc':  roc_auc_score(yt, oof_lgb[m]),
        'xgb_auc':  roc_auc_score(yt, oof_xgb[m]),
        'rmlp_auc': roc_auc_score(yt, oof_rm[m]),
        'cb_pred_mean': float(oof_cb[m].mean()),
    })
yr_df = pd.DataFrame(rows).set_index('year')
print(yr_df.round(4))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
yr_df[['lgb_auc','cb_auc','xgb_auc','rmlp_auc']].plot.bar(ax=axes[0])
axes[0].set_title('Per-year OOF AUC'); axes[0].set_ylim(0.85, 1.0)
yr_df[['pit_rate','cb_pred_mean']].plot.bar(ax=axes[1])
axes[1].set_title('Per-year base rate vs CB mean pred (calibration sanity)')
plt.tight_layout(); plt.show()

In [ ]:
# 7c. Logistic stacker — Ivan's negative-coefficient detector
# Use logit of each model's OOF probability so the linear model is well-conditioned.
models = {'lgb':oof_lgb, 'cb_bag':oof_cb, 'xgb':oof_xgb, 'realmlp':oof_rm}
stack_X = np.column_stack([logit(np.clip(o, 1e-6, 1-1e-6)) for o in models.values()])
stack_te = {'lgb':pred_lgb, 'cb_bag':pred_cb, 'xgb':pred_xgb, 'realmlp':pred_rm}
stack_X_te = np.column_stack([logit(np.clip(p, 1e-6, 1-1e-6)) for p in stack_te.values()])
lr = LogisticRegression(C=10, max_iter=500).fit(stack_X, y)
coefs = dict(zip(models.keys(), lr.coef_[0]))
print('Stack coefficients (logit-space):')
for k,v in coefs.items(): print(f'  {k:>10s}: {v:+.4f}{"  ← NEGATIVE: drop" if v<0 else ""}')
drop = {k for k,v in coefs.items() if v < 0}
kept = [k for k in models if k not in drop]
print(f'\nKeeping: {kept}')
auc_stack = roc_auc_score(y, lr.predict_proba(stack_X)[:,1])
print(f'LR stacker OOF AUC: {auc_stack:.5f}')

In [ ]:
# 7d. Per-race AUC (weakest 20) — bagged CB
race_auc = (train_fe.assign(oof=oof_cb).groupby('Race')
            .apply(lambda d: roc_auc_score(d[TARGET], d['oof']) if d[TARGET].nunique()>1 else np.nan)
            .sort_values())
fig, ax = plt.subplots(figsize=(8, 8))
race_auc.head(20).plot.barh(ax=ax, color='steelblue')
ax.axvline(auc_cb, color='red', ls='--', label=f'overall {auc_cb:.4f}')
ax.set_title('20 weakest races (bagged CB OOF)'); ax.set_xlabel('AUC'); ax.legend()
plt.tight_layout(); plt.show()

## 8. Final blend + isotonic calibration

In [ ]:
# Option A: logit-rank across all models (legacy)
all_oof  = [oof_lgb, oof_cb, oof_xgb, oof_rm]
all_pred = [pred_lgb, pred_cb, pred_xgb, pred_rm]
all_names = ['lgb','cb_bag','xgb','realmlp']
all_aucs  = [auc_lgb, auc_cb, auc_xgb, auc_rm]
w_all = np.array(all_aucs) ** 2
oof_full  = logit_rank(all_oof,  w_all)
pred_full = logit_rank(all_pred, w_all)
auc_full  = roc_auc_score(y, oof_full)

# Option B: drop models with negative stacker coefs, then logit-rank
kept_oof  = [o for n,o in zip(all_names, all_oof)  if n in kept]
kept_pred = [p for n,p in zip(all_names, all_pred) if n in kept]
kept_aucs = [a for n,a in zip(all_names, all_aucs) if n in kept]
w_kept = np.array(kept_aucs) ** 2
oof_kept  = logit_rank(kept_oof,  w_kept) if kept_oof else None
pred_kept = logit_rank(kept_pred, w_kept) if kept_pred else None
auc_kept  = roc_auc_score(y, oof_kept) if oof_kept is not None else 0.0

# Option C: LR stacker preds
oof_lr  = lr.predict_proba(stack_X)[:,1]
pred_lr = lr.predict_proba(stack_X_te)[:,1]

results = {
    'lgb': (auc_lgb, pred_lgb),
    'cb_bag': (auc_cb, pred_cb),
    'xgb': (auc_xgb, pred_xgb),
    'realmlp': (auc_rm, pred_rm),
    'blend_full': (auc_full, pred_full),
    'blend_kept': (auc_kept, pred_kept) if pred_kept is not None else (0.0, None),
    'stacker_lr': (auc_stack, pred_lr),
}
for n, (a, _) in results.items():
    print(f'  {n:>12s}: OOF AUC = {a:.5f}')
best_name = max((k for k,(a,p) in results.items() if p is not None), key=lambda k: results[k][0])
best_auc, best_pred = results[best_name]
print(f'\nBest: {best_name} @ {best_auc:.5f}')

# Isotonic calibration on best OOF, applied to best test preds (preserves AUC)
best_oof = {
    'lgb': oof_lgb, 'cb_bag': oof_cb, 'xgb': oof_xgb, 'realmlp': oof_rm,
    'blend_full': oof_full, 'blend_kept': oof_kept, 'stacker_lr': oof_lr,
}[best_name]
iso = IsotonicRegression(out_of_bounds='clip').fit(best_oof, y)
final_pred = iso.transform(best_pred)
print(f'After isotonic: pred mean={final_pred.mean():.4f} (train pos rate={y.mean():.4f})')

In [ ]:
# Reliability diagram — sanity check that calibration moved the predictions onto the diagonal
prob_true_raw,  prob_pred_raw  = calibration_curve(y, best_oof, n_bins=20)
prob_true_iso,  prob_pred_iso  = calibration_curve(y, iso.transform(best_oof), n_bins=20)
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0,1],[0,1],'k--', alpha=0.5, label='perfect')
ax.plot(prob_pred_raw, prob_true_raw, 'o-', label='raw', color='steelblue')
ax.plot(prob_pred_iso, prob_true_iso, 's-', label='isotonic', color='darkorange')
ax.set_xlabel('Predicted'); ax.set_ylabel('Empirical rate')
ax.set_title(f'Reliability diagram ({best_name})'); ax.legend()
plt.tight_layout(); plt.show()

## 9. Submission

In [ ]:
sub_out = test_fe[[ID]].copy()
sub_out[TARGET] = final_pred
sub_out.to_csv('submission.csv', index=False)
print(f'wrote submission.csv  shape={sub_out.shape}  pred mean={final_pred.mean():.4f}')
sub_out.head()